# 🚗 DriveSense: AI-Driven In-Vehicle Digital Voice Assistant
### Intelligence & Intent-Understanding Layer for Connected Vehicles
---
**DriveSense** is designed as a decoupled, modular AI assistant for in-vehicle environments:
1. **Voice Input (STT)**: Offline speech recognition via **Vosk**.
2. **Intelligence Layer (DriveSense AI)**: Classifies driver requests into structured JSON intents (`GREETING`, `GET_TEMPERATURE`, `GET_DISTANCE`, `GENERAL_QUERY`, `EXIT`, `UNSUPPORTED`).
3. **Application & Hardware Dispatcher**: Executes real sensor readings (DHT cabin temperature sensor & HC-SR04 ultrasonic distance sensor) or simulation on laptops.
4. **Voice Output (TTS)**: Delivers concise, safe spoken responses via Text-to-Speech.

## 1. Speech Recognition & Interactive Voice Input

In [ ]:
# Import Core Libraries
import os
import re
import json
import time
import threading
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

# Audio and Speech Recognition
import pyaudio
from vosk import Model, KaldiRecognizer

# Text-to-Speech
try:
    import pyttsx3
    HAS_TTS = True
except ImportError:
    HAS_TTS = False

print("All core DriveSense modules imported successfully!")

In [ ]:
# Initialize Offline Vosk Speech Recognition Model
FRAME_RATE = 16000
model_path = "vosk-model-small-en-in-0.4"
if not os.path.exists(model_path):
    if os.path.exists("vosk-model-small-en-us-0.15"):
        model_path = "vosk-model-small-en-us-0.15"

print(f"Loading Vosk Model from '{model_path}'...")
vosk_model = Model(model_path)
rec = KaldiRecognizer(vosk_model, FRAME_RATE)
rec.SetWords(True)
print("Vosk Speech Recognizer initialized successfully!")

In [ ]:
# Real-Time Interactive Microphone Recording UI
is_recording = False
active_driver_command = "What is the temperature?"

record_button = widgets.Button(
    description='Start Recording',
    button_style='success',
    tooltip='Click to speak into microphone',
    icon='microphone'
)

stop_button = widgets.Button(
    description='Stop & Recognize',
    button_style='danger',
    disabled=True,
    tooltip='Click when done speaking',
    icon='stop'
)

text_input_box = widgets.Text(
    value="What is the temperature?",
    placeholder="Or type a command here (e.g. Is there an obstacle ahead?)",
    description='Command:',
    layout=widgets.Layout(width='60%')
)

output_status = widgets.Output()

def audio_stream_loop():
    global is_recording, active_driver_command
    p = pyaudio.PyAudio()
    transcribed_segments = []
    try:
        stream = p.open(
            format=pyaudio.paInt16,
            channels=1,
            rate=FRAME_RATE,
            input=True,
            frames_per_buffer=2048
        )
        while is_recording:
            data = stream.read(2048, exception_on_overflow=False)
            if len(data) > 0:
                if rec.AcceptWaveform(data):
                    res = json.loads(rec.Result())
                    txt = res.get("text", "").strip()
                    if txt:
                        transcribed_segments.append(txt)
                        with output_status:
                            print(f"  Listening... -> {txt}")
        
        final_res = json.loads(rec.FinalResult())
        final_txt = final_res.get("text", "").strip()
        if final_txt:
            transcribed_segments.append(final_txt)
            
        full_command = " ".join(transcribed_segments).strip()
        if full_command:
            active_driver_command = full_command
            text_input_box.value = full_command
            with output_status:
                print("="*50)
                print(f"VOICE TRANSCRIBED: '{full_command}'")
                print("="*50)
        else:
            with output_status:
                print("[No clear speech detected. Using text input box command.]")
                
        stream.stop_stream()
        stream.close()
    except Exception as e:
        with output_status:
            print(f"Microphone notice: {e}")
    finally:
        p.terminate()

def start_rec_clicked(btn):
    global is_recording
    if not is_recording:
        is_recording = True
        record_button.disabled = True
        stop_button.disabled = False
        with output_status:
            output_status.clear_output()
            print("Recording in progress... Speak your in-vehicle command now:")
        t = threading.Thread(target=audio_stream_loop)
        t.start()

def stop_rec_clicked(btn):
    global is_recording
    if is_recording:
        is_recording = False
        record_button.disabled = False
        stop_button.disabled = True
        with output_status:
            print("Processing speech recognition...")

record_button.on_click(start_rec_clicked)
stop_button.on_click(stop_rec_clicked)

ui_container = widgets.VBox([
    widgets.HBox([record_button, stop_button]),
    text_input_box,
    output_status
])
display(ui_container)

## 2. DriveSense AI: Intelligence & Intent-Understanding Layer
Classifies input into one of the 6 official DriveSense intents and returns **strictly structured JSON** without ever fabricating sensor data.

In [ ]:
from drivesense import DriveSenseAI, DriveSenseHardwareDispatcher

ai_engine = DriveSenseAI()
hardware_dispatcher = DriveSenseHardwareDispatcher(is_raspberry_pi=False)

# Process active driver input through the AI Layer
current_query = text_input_box.value.strip() if text_input_box.value.strip() else active_driver_command
json_intent_output = ai_engine.generate_response(current_query)

print(f"Driver Query: '{current_query}'")
print("\n--- [DriveSense AI Structured JSON Output] ---")
print(json.dumps(json_intent_output, indent=2))

## 3. Hardware Sensor Dispatcher & Text-to-Speech Output
The Python application receives the structured JSON from the AI Layer and performs actual sensor reading or hardware interaction.

In [ ]:
# Execute Sensor Reading & Dispatch
final_spoken_output = hardware_dispatcher.dispatch(json_intent_output)

print(f"Intent Executed:     {json_intent_output['intent']}")
print(f"Sensor/Action Value: {final_spoken_output}")

# Optional TTS Output
if HAS_TTS:
    try:
        engine = pyttsx3.init()
        engine.setProperty('rate', 170)
        engine.say(final_spoken_output)
        engine.runAndWait()
    except Exception as e:
        pass

## 4. DriveSense Machine Learning Intent Classification Pipeline
Training & evaluating ML Classifiers (**SVC, Logistic Regression, Random Forest, MLP, Gradient Boosting**) across the 6 DriveSense intent classes.

In [ ]:
# Load DriveSense Intent Dataset
df = pd.read_csv("drivesense_dataset.csv")
print(f"DriveSense Dataset: {df.shape[0]} labeled samples across {df['Label'].nunique()} intents:")
print(df['Label'].value_counts())
display(df.head(10))

In [ ]:
# Text Preprocessing & Vectorization
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

def clean_sentence(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = [w for w in text.split() if w.strip()]
    stop_words = set(stopwords.words('english')) - {'not', 'no', 'is', 'there', 'what', 'how'}
    words = [w for w in words if w not in stop_words]
    lemmatizer = WordNetLemmatizer()
    return ' '.join([lemmatizer.lemmatize(w) for w in words])

df['Clean_Sentence'] = df['Sentence'].apply(clean_sentence)

label_enc = LabelEncoder()
y_encoded = label_enc.fit_transform(df['Label'])

vec = CountVectorizer(ngram_range=(1, 2))
X_vec = vec.fit_transform(df['Clean_Sentence'])

X_train, X_test, y_train, y_test = train_test_split(X_vec, y_encoded, test_size=0.25, random_state=42, stratify=y_encoded)
print(f"Training set shape: {X_train.shape}")
print(f"Test set shape:     {X_test.shape}")

In [ ]:
# Model Training & Evaluation
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

models = [
    ("SVC (Linear)", SVC(C=1, kernel='linear', probability=True, random_state=42)),
    ("Logistic Regression", LogisticRegression(max_iter=500, random_state=42)),
    ("MLP Neural Net", MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=42)),
    ("Random Forest", RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)),
    ("Gradient Boosting", GradientBoostingClassifier(random_state=42))
]

print(f"{'Model':<24} | {'Test Accuracy':<14} | {'Weighted F1':<12}")
print("-" * 56)

best_clf = None
best_acc = 0

for name, clf in models:
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    print(f"{name:<24} | {acc:.4f}{' '*8} | {f1:.4f}")
    if acc > best_acc:
        best_acc = acc
        best_clf = clf

y_test_pred = best_clf.predict(X_test)
print("\n--- Best Model Classification Report ---")
print(classification_report(y_test, y_test_pred, target_names=label_enc.classes_))

In [ ]:
# Confusion Matrix Visualization
cm = confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(9, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_enc.classes_, yticklabels=label_enc.classes_)
plt.title('DriveSense Intent Classification Confusion Matrix', fontsize=14)
plt.xlabel('Predicted Intent', fontsize=12)
plt.ylabel('Actual Intent', fontsize=12)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 5. Live DriveSense Multi-Intent Test Suite
Validates all 6 intents across edge cases and unsupported vehicle requests.

In [ ]:
test_suite = [
    "Hello DriveSense",
    "What is the cabin temperature?",
    "Is there an obstacle in front of me?",
    "What is artificial intelligence?",
    "Call my friend.",
    "Navigate to Chennai.",
    "What is my fuel level?",
    "Turn on the AC.",
    "Detect whether I am sleepy.",
    "Goodbye"
]

print("="*75)
print(f"{'INPUT PROMPT':<35} | {'CLASSIFIED INTENT':<18} | {'DELIVERED ACTION'}")
print("="*75)

for prompt in test_suite:
    payload = ai_engine.generate_response(prompt)
    delivered = hardware_dispatcher.dispatch(payload)
    print(f"{prompt:<35} | {payload['intent']:<18} | {delivered}")